# 03 — Customer Master Table & RFM Analysis

This notebook is the current active practice stage.

**Goal:** reduce transaction-level data to one customer-level row and create interpretable Recency, Frequency and Monetary features.


In [ ]:
order_payments = pd.read_csv(
    "../data_raw/olist_order_payments_dataset.csv"
)

order_value = (
    order_payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_value"})
)

customer_orders_value = customer_orders.merge(
    order_value,
    on="order_id",
    how="left"
)


In [ ]:
customer_master = (
    customer_orders_value
    .groupby("customer_unique_id")
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("order_value", "sum")
    )
    .reset_index()
)

customer_master["customer_lifetime_days"] = (
    customer_master["last_purchase"]
    - customer_master["first_purchase"]
).dt.days

customer_master.shape


Observed customer-master grain: **96,096 rows = 96,096 unique customers**.


In [ ]:
analysis_date = (
    customer_master["last_purchase"].max()
    + pd.Timedelta(days=1)
)

customer_master["recency"] = (
    analysis_date - customer_master["last_purchase"]
).dt.days

customer_master[["recency", "frequency", "monetary"]].describe()


## RFM interpretation

- **Recency:** lower is better
- **Frequency:** higher is better
- **Monetary:** higher is better

Frequency is extremely concentrated at 1 order, so a business-rule score is used instead of blindly forcing quintiles.


In [ ]:
customer_master["R_score"] = pd.qcut(
    customer_master["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

customer_master["M_score"] = pd.qcut(
    customer_master["monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x <= 5:
        return 4
    return 5

customer_master["F_score"] = customer_master["frequency"].apply(frequency_score)

customer_master["RFM_score"] = (
    customer_master["R_score"].astype(str)
    + customer_master["F_score"].astype(str)
    + customer_master["M_score"].astype(str)
)

customer_master["RFM_total"] = (
    customer_master["R_score"]
    + customer_master["F_score"]
    + customer_master["M_score"]
)

customer_master[[
    "customer_unique_id", "recency", "frequency", "monetary",
    "R_score", "F_score", "M_score", "RFM_score", "RFM_total"
]].head(10)


## Next practice step

Validate the RFM score distribution, then map scores into business segments such as Champions, Loyal Customers, New Customers, At Risk, and Lost.
